## 1. Leer los datos


Primero importamos `pandas` y leemos directamente el archivo.

In [ ]:
import pandas as pd

# URL del archivo original, publicado en el repositorio del curso en GitHub
url = "https://github.com/doctorado-cuanticp/rdit/raw/main/rdit_dataoriginal.csv"

rdit = pd.read_csv(url)

rdit.head()
# Debe mostrar 6 columnas: id_proyecto, fecha_votacion, fecha_declaracion,
# pct_apoyo_oposicion, asistencia_pleno y tramite_urgencia.

## 2. Revisar la estructura de los datos

Antes de transformar nada, revisamos qué variables tenemos y qué tipo de información contiene cada una.

In [ ]:
rdit.info()
# Observe que fecha_votacion y fecha_declaracion aparecen como 'object' (texto),
# no como fechas. Eso es lo que corregimos en el siguiente paso.

Todavía no tenemos dos variables fundamentales para el diseño:

- `dias_relativos`
- `post_declaracion`

Las construiremos a partir de `fecha_votacion` y `fecha_declaracion`.

## 3. Convertir las fechas

Python debe reconocer las dos columnas como fechas antes de que podamos calcular distancias entre ellas.

In [ ]:
rdit["fecha_votacion"] = pd.to_datetime(rdit["fecha_votacion"])
rdit["fecha_declaracion"] = pd.to_datetime(rdit["fecha_declaracion"])

rdit.dtypes
# Ahora ambas columnas deben aparecer como 'datetime64[ns]'.
# Sin esta conversión, la resta de fechas del siguiente paso no funcionaría
# (o daría un resultado sin sentido, porque Python restaría texto, no fechas).

## 4. Construir la variable temporal

Calculamos cuántos días separan cada votación de la fecha de las declaraciones presidenciales.

Interpretación:

- un valor negativo significa que la votación ocurrió **antes**;
- `0` corresponde al día de las declaraciones;
- un valor positivo significa que la votación ocurrió **después**.

In [ ]:
rdit["dias_relativos"] = (
    rdit["fecha_votacion"] - rdit["fecha_declaracion"]
).dt.days

rdit[
    ["fecha_votacion", "fecha_declaracion", "dias_relativos"]
].head(10)
# El orden de la resta importa: votacion menos declaracion.
# Negativo = la votación fue antes de las declaraciones.
# Positivo = la votación fue después.

Por ejemplo:

- `dias_relativos = -10` significa 10 días antes;
- `dias_relativos = 10` significa 10 días después.

La fecha de las declaraciones es fija. Lo que cambia entre observaciones es la distancia de cada votación respecto de esa fecha.

## 5. Construir el indicador antes/después

Ahora construimos una variable que vale:

- `0` antes de las declaraciones;
- `1` desde el día de las declaraciones en adelante.

In [ ]:
rdit["post_declaracion"] = (
    rdit["dias_relativos"] >= 0
).astype(int)

rdit[
    ["dias_relativos", "post_declaracion"]
].head(15)
# >= 0 incluye el día mismo de las declaraciones dentro del grupo "post".
# post_declaracion es la variable de tratamiento que usará RDiT-04 para
# estimar el cambio inmediato alrededor del punto de corte.

In [ ]:
# fecha_votacion y fecha_declaracion ya cumplieron su función: sirvieron para
# calcular dias_relativos. El notebook de análisis (RDiT) no vuelve a usarlas,
# así que no hace falta conservarlas ni preservar su tipo datetime.
# Por eso guardamos en CSV (más simple y legible que pickle) y sin las fechas.
rdit_final = rdit.drop(columns=["fecha_votacion", "fecha_declaracion"])

rdit_final.to_csv('rdit_preparado.csv', index=False)
rdit_final.head()